## benchmark

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< sm'_operators

## fsharp

In [1]:
:== test_case_result =
    {
        input : string
        expected : string
        result : string
        time_list : array_base i64
    }

:= run' \* t. count (fn : () -> t) =
    process::gc_collect ()
    := stopwatch = date_time::stopwatch ()
    stopwatch |> date_time::stopwatch_start
    := time1 = stopwatch |> date_time::stopwatch_elapsed_milliseconds
    := result : t =
        am'::init_series 0 count 1i32
        |> \ x => array x : array _
        |> am'::parallel_map \ _n => fn ()
        |> am'::last
    := time2 = (stopwatch |> date_time::stopwatch_elapsed_milliseconds) - time1
    result, time2

:= run \* input expected.
    count
    (solutions : list (string * (input -> expected)))
    ((input, expected) : (input * expected))
    : test_case_result
    =
    := input_str = #?*input

    console::write_line ""
    trace Verbose
        \ () => "benchmark.run"
        \ () => { input_str = input_str |> sm'::ellipsis_end 40 }

    := results_with_time : array_base _ =
        solutions
        |> listm'::indexed
        |> listm'::box
        |> listm'::to_array'
        |> am'::map_base \ ((i : int), (test_name, solution)) =>
            := result, time =
                \ () => solution input
                |> run' count
            trace Verbose
                \ () => "benchmark.run / solutions.map"
                \ () => { i = i + 1; test_name time }
            result, time

    ?= results_with_time |> am'::map_base fst ?|
    | items ?& (items |> (\ x => array x : array _) |> am'::length) <= 1 => ()
    | items ?&
        items
        |> (\ x => array x : array _)
        |> arraym::forall' ((=) (items |> (\ x => array x : array _) |> am'::index 0))
        => ()
    | results => failwith ($'$"benchmark.run / error / results: {!results}"' : string)

    {
        input = input_str
        expected = #?*expected
        result =
            results_with_time
            |> am'::map_base fst
            |> (\ x => array x : array _)
            |> am'::index 0
            |> sm'::format_debug
        time_list = results_with_time |> am'::map_base snd
    }

:= run_all \* input expected.
    test_name
    count
    (solutions : list (string * (input -> expected)))
    test_cases
    =
    console::write_line ""
    console::write_line "```"
    trace Verbose
        \ () => "benchmark.run_all"
        \ () => { test_name count }
    test_cases
    |> listm'::box
    |> listm'::to_array'
    |> am'::map_base (run count solutions)

:= sort_result_list results =
    := table =
        := rows =
            results
            |> am'::map_base \ (result : test_case_result) =>
                := best =
                    result::time_list
                    |> am'::indexed
                    |> am'::map_base \ (i, time) =>
                        i + 1i32, time
                    |> \ x => array x : array _
                    |> am'::sort_by snd
                    |> am'::index 0i32
                    |> sm'::format
                := row =
                    [
                        result::input |> sm'::ellipsis_end 40 |> sm'::replace "|" ""
                        result::expected
                        result::result
                        best
                    ]
                := color : option console::console_color =
                    :< console
                    ?= result::expected = result::result ?|
                    | !+ => Some $'`console_color.DarkGreen'
                    | !- => Some $'`console_color.DarkRed'
                row, color

        := header =
            [
                [
                    "input"
                    "expected"
                    "result"
                    "best"
                ]
                [
                    "---"
                    "---"
                    "---"
                    "---"
                ]
            ]
            |> listm::map \ row => row, None
            |> listm'::box
            |> listm'::to_array'
            |> \ x => array x : array _
        array rows
        |> arraym::append header
        |> \ (array x) => x

    := formatted_table =
        := length_map : map::map i32 i64 =
            table
            |> am'::map_base (fst >> listm'::box >> listm'::to_array')
            |> am'::transpose
            |> am'::map_base \ column =>
                column
                |> am'::map_base sm::length
                |> \ x => array x : array _
                |> am'::sort_descending
                |> am'::try_item 0i32
                |> optionm'::default_value 0i64
            |> am'::indexed
            |> \ x => array x : array _
            |> map::of_array
        table
        |> am'::map_base \ (row, color) =>
            := new_row =
                row
                |> listm'::indexed
                |> listm::map \ (i, cell) =>
                    cell |> sm'::pad_right (length_map |> map::item i |> conv) ' '
                |> listm'::box
                |> listm'::to_array'
            new_row, color

    console::write_line "```"
    formatted_table
    |> \ x => array x : array _
    |> am'::to_list'
    |> listm'::unbox
    |> listm::iter \ (row, color) =>
        ?= color ?|
        | Some color => color |> console::set_foreground_color
        | None => console::reset_color ()

        array row |> sm'::join' "\t| " |> console::write_line

        console::reset_color ()

    := averages =
        results
        |> am'::map_base \ result =>
            result::time_list
            |> am'::map_base ($'float' : i64 -> f64)
        |> am'::transpose
        |> am'::map_base ((\ x => array x : array _) >> am'::average)
        |> am'::map_base ($'int64' : f64 -> i64)
        |> am'::indexed
        |> \ x => array x : array _

    console::write_line "```"
    averages
    |> am'::sort_by snd
    |> am'::to_list'
    |> listm'::unbox
    |> listm::iter \ ((i : i32), avg) =>
        trace Verbose
            \ () => "benchmark.sort_result_list / averages.iter"
            \ () => { i = i + 1; avg }
    console::write_line "```"

#### sort_result_list (test)

In [ ]:
///- --test static --symbol

:= is_fast () =
    !-

### empty2Tests

Test: Empty2

Solution: (a, a)
Test case 1. A. Time: 59L

Solution: (a, a)
Test case 1. A. Time: 53L

Input   | Expected        | Result  | Best
---     | ---             | ---     | ---
(a, a)  | a               | a       | (1, 59)
(a, a)  | a               | a       | (1, 53)

Averages
Test case 1. Average Time: 56L

Ranking
Test case 1. Average Time: 56L

---


```
01:12:03 [90mverbose[0m #1 benchmark.run_all / {count = 2000000; test_name = empty_2_tests}
01:12:03 [90mverbose[0m #2 benchmark.run / {count = 2000000; expected = a; input = a, a; input_str = struct ("a", "a")}
01:12:03 [90mverbose[0m #3 benchmark.run / solutions.map / {count = 2000000; expected = a; i = 0; input = a, a; input_str = struct ("a", "a"); test_name = A; time = 119}
01:12:04 [90mverbose[0m #4 benchmark.run / solutions.map / {count = 2000000; expected = a; i = 1; input = a, a; input_str = struct ("a", "a"); test_name = B; time = 122}
01:12:04 [90mverbose[0m #5 benchmark.run / {count = 2000000; expected = b; input = b, b; input_str = struct ("b", "b")}
01:12:04 [90mverbose[0m #6 benchmark.run / solutions.map / {count = 2000000; expected = b; i = 0; input = b, b; input_str = struct ("b", "b"); test_name = A; time = 110}
01:12:04 [90mverbose[0m #7 benchmark.run / solutions.map / {count = 2000000; expected = b; i = 1; input = b, b; input_str = struct ("b", "b"); test_name = B; time = 120}
```
Input            	| Expected	| Result	| Best
---              	| ---     	| ---   	| ---
struct ("a", "a")	| "a"     	| "a"   	| struct (1L, 119L)
struct ("b", "b")	| "b"     	| "b"   	| struct (1L, 110L)
```
01:12:04 [90mverbose[0m #8 benchmark.sort_result_list / averages.iter / {avg = 114; i = 0}
01:12:04 [90mverbose[0m #9 benchmark.sort_result_list / averages.iter / {avg = 121; i = 1}
```
`

In [2]:
///- --test --symbol
///> fsharp

:= get_solutions () =
    [
        "A",
        \ (a, _b) =>
            a

        "B",
        \ (_a, b) =>
            b
    ]

:= ^ empty_2_tests () =
    := test_cases = [
        ("a", "a"), "a"
        ("b", "b"), "b"
    ]

    := solutions = get_solutions ()


    := count =
        ? is_fast ()
        ?> 1000i32
        |: 2000000i32

    run_all (reflection::nameof { empty_2_tests }) count solutions test_cases
    |> sort_result_list

empty_2_tests ()

```
17:40:19 1v benchmark.run_all / { test_name = empty_2_tests; count = 2000000 }
17:40:19 2v benchmark.run / { input_str = struct ("a", "a") }
17:40:19 3v benchmark.run / solutions.map / { i = 1; test_name = A; time = 55 }
17:40:19 4v benchmark.run / solutions.map / { i = 2; test_name = B; time = 23 }
17:40:19 5v benchmark.run / { input_str = struct ("b", "b") }
17:40:19 6v benchmark.run / solutions.map / { i = 1; test_name = A; time = 16 }
17:40:19 7v benchmark.run / solutions.map / { i = 2; test_name = B; time = 19 }
```
input            	| expected	| result	| best   
---              	| ---     	| ---   	| ---    
struct ("a", "a")	| a       	| a     	| (2, 23)
struct ("b", "b")	| b       	| b     	| (1, 16)
```
17:40:19 8v benchmark.sort_result_list / averages.iter / { i = 2; avg = 21 }
17:40:19 9v benchmark.sort_result_list / averages.iter / { i = 1; avg = 35 }
```

### emptyTests

Test: Empty

Solution: 0
Test case 1. A. Time: 61L

Solution: 2
Test case 1. A. Time: 62L

Solution: 5
Test case 1. A. Time: 70L

Input   | Expected        | Result  | Best
---     | ---             | ---     | ---
0       | 0               | 0       | (1, 61)
2       | 2               | 2       | (1, 62)
5       | 5               | 5       | (1, 70)

Averages
Test case 1. Average Time: 64L

Ranking
Test case 1. Average Time: 64L

---

```
01:21:25 [90mverbose[0m #1 benchmark.run_all / {count = 2000000; test_name = empty_1_tests}
01:21:25 [90mverbose[0m #2 benchmark.run / {count = 2000000; expected = +1.000000; input = +0.000000; input_str = 0.0}
01:21:25 [90mverbose[0m #3 benchmark.run / solutions.map / {count = 2000000; expected = +1.000000; i = 0; input = +0.000000; input_str = 0.0; test_name = A; time = 36}
01:21:25 [90mverbose[0m #4 benchmark.run / {count = 2000000; expected = +3.000000; input = +2.000000; input_str = 2.0}
01:21:25 [90mverbose[0m #5 benchmark.run / solutions.map / {count = 2000000; expected = +3.000000; i = 0; input = +2.000000; input_str = 2.0; test_name = A; time = 20}
01:21:25 [90mverbose[0m #6 benchmark.run / {count = 2000000; expected = +6.000000; input = +5.000000; input_str = 5.0}
01:21:25 [90mverbose[0m #7 benchmark.run / solutions.map / {count = 2000000; expected = +6.000000; i = 0; input = +5.000000; input_str = 5.0; test_name = A; time = 22}
```
Input	| Expected	| Result	| Best
---  	| ---     	| ---   	| ---
0.0  	| 1.0     	| 1.0   	| struct (1L, 36L)
2.0  	| 3.0     	| 3.0   	| struct (1L, 20L)
5.0  	| 6.0     	| 6.0   	| struct (1L, 22L)
```
01:21:25 [90mverbose[0m #8 benchmark.sort_result_list / averages.iter / {avg = 26; i = 0}
```

In [3]:
///- --test --symbol
///> fsharp

:= get_solutions () =
    [
        "A",
        \ n =>
            n + 1f64
    ]

:= ^ empty_1_tests () =
    := test_cases = [
        0, 1
        2, 3
        5, 6
    ]

    := solutions = get_solutions ()


    := count =
        ? is_fast ()
        ?> 1000i32
        |: 2000000i32

    run_all (reflection::nameof { empty_1_tests }) count solutions test_cases
    |> sort_result_list

empty_1_tests ()

```
17:40:20 1v benchmark.run_all / { test_name = empty_1_tests; count = 2000000 }
17:40:20 2v benchmark.run / { input_str = 0.0 }
17:40:20 3v benchmark.run / solutions.map / { i = 1; test_name = A; time = 43 }
17:40:20 4v benchmark.run / { input_str = 2.0 }
17:40:20 5v benchmark.run / solutions.map / { i = 1; test_name = A; time = 15 }
17:40:20 6v benchmark.run / { input_str = 5.0 }
17:40:20 7v benchmark.run / solutions.map / { i = 1; test_name = A; time = 14 }
```
input	| expected	| result	| best   
---  	| ---     	| ---   	| ---    
0.0  	| 1.0     	| 1.0   	| (1, 43)
2.0  	| 3.0     	| 3.0   	| (1, 15)
5.0  	| 6.0     	| 6.0   	| (1, 14)
```
17:40:20 8v benchmark.sort_result_list / averages.iter / { i = 1; avg = 24 }
```